In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.3f}".format)

DATA_RAW       = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

In [2]:
train = pd.read_csv(DATA_RAW / "application_train.csv")
test  = pd.read_csv(DATA_RAW / "application_test.csv")

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")

Train : (307511, 122)
Test  : (48744, 121)


## Stratégie

On travaille en parallèle sur train et test pour appliquer
les mêmes transformations aux deux datasets.
On les fusionne temporairement pour le feature engineering,
puis on les sépare avant la sauvegarde.

In [3]:
train = train.copy()
test  = test.copy()

train["is_train"] = 1
test["is_train"]  = 0 # Ajoute une colonne booléenne pour se souvenir quelle ligne vient du train et laquelle vient du test. On en aura besoin à la fin pour re-séparer les deux datasets après le feature engineering.
test["TARGET"]    = np.nan # Ajoute une colonne TARGET dans le test pour faciliter la concaténation. On la remplira de NaN pour se souvenir que ce sont des données de test sans étiquette.

df = pd.concat([train, test], ignore_index=True)
df = df.copy()  # défragmente
print(f"Dataset combiné : {df.shape}")

Dataset combiné : (356255, 123)


## Data quality Audit

In [4]:
# Aperçu des colonnes DAYS_* — ce sont des variables temporelles exprimées en nombre de jours avant la demande de crédit. 
# Par exemple, DAYS_BIRTH est l'âge du client en jours (valeurs négatives), DAYS_EMPLOYED est le nombre de jours depuis l'embauche (valeurs négatives, mais 365 jours = 1 an), etc.
days_cols = [c for c in df.columns if c.startswith("DAYS_")]
print(df[days_cols].describe().T[["min", "max", "mean", "50%"]])

                              min        max       mean        50%
DAYS_BIRTH             -25229.000  -7338.000 -16041.249 -15755.000
DAYS_EMPLOYED          -17912.000 365243.000  64317.231  -1224.000
DAYS_REGISTRATION      -24672.000      0.000  -4983.594  -4502.000
DAYS_ID_PUBLISH         -7197.000      0.000  -3002.071  -3252.000
DAYS_LAST_PHONE_CHANGE  -4361.000      0.000   -978.581   -771.000


Le pattern est immédiatement visible.

- DAYS_BIRTH — min -25229, max -7338. Toutes les valeurs sont négatives et dans une plage cohérente (environ 20 à 69 ans). Pas d'anomalie.
- DAYS_REGISTRATION, DAYS_ID_PUBLISH, DAYS_LAST_PHONE_CHANGE — toutes négatives, max à 0. Cohérent, pas d'anomalie.
- DAYS_EMPLOYED — min -17912 (environ 49 ans d'ancienneté, plausible), mais max à 365 243. C'est la seule valeur positive et elle est absurde — ça représenterait ~1000 ans d'emploi. C'est clairement une valeur sentinelle.

In [5]:
# Combien de clients ont cette valeur ?
mask = df["DAYS_EMPLOYED"] == 365243
print(f"Valeurs 365243 : {mask.sum()} ({mask.mean()*100:.1f}%)")

# Quel est leur profil TARGET ?
print("\nTaux de défaut selon DAYS_EMPLOYED :")
print(df.groupby(mask)["TARGET"].mean().rename({False: "Valeur normale", True: "365243"}))

# Quel type de revenu ont ces clients ?
print("\nNAME_INCOME_TYPE pour ces clients :")
print(df[mask]["NAME_INCOME_TYPE"].value_counts())

Valeurs 365243 : 64648 (18.1%)

Taux de défaut selon DAYS_EMPLOYED :
DAYS_EMPLOYED
Valeur normale   0.087
365243           0.054
Name: TARGET, dtype: float64

NAME_INCOME_TYPE pour ces clients :
NAME_INCOME_TYPE
Pensioner     64625
Unemployed       23
Name: count, dtype: int64


In [6]:
# Traitement de l'anomalie DAYS_EMPLOYED = 365243
# Créer un flag avant de remplacer — la valeur est informative
df["DAYS_EMPLOYED_ANOMALY"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)

# Remplacer par NaN — sera imputé plus tard avec la médiane
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

# Vérification
print(f"Flag DAYS_EMPLOYED_ANOMALY : {df['DAYS_EMPLOYED_ANOMALY'].sum()} clients")
print(f"NaN DAYS_EMPLOYED après remplacement : {df['DAYS_EMPLOYED'].isna().sum()}")
print(f"\nDAYS_EMPLOYED après nettoyage :")
print(df["DAYS_EMPLOYED"].describe())

Flag DAYS_EMPLOYED_ANOMALY : 64648 clients
NaN DAYS_EMPLOYED après remplacement : 64648

DAYS_EMPLOYED après nettoyage :
count   291607.000
mean     -2396.699
std       2334.480
min     -17912.000
25%      -3200.000
50%      -1663.000
75%       -780.000
max          0.000
Name: DAYS_EMPLOYED, dtype: float64


In [7]:
# Traitement des features temporelles
# Convertir les DAYS_ en valeurs positives et lisibles
# DAYS_BIRTH : âge en années
df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365).astype(int)

# DAYS_EMPLOYED : ancienneté en années (NaN pour les retraités)
df["EMPLOYED_YEARS"] = (-df["DAYS_EMPLOYED"] / 365)

# Ratio âge/ancienneté : part de la vie active passée chez l'employeur actuel
df["EMPLOYMENT_TO_AGE_RATIO"] = df["EMPLOYED_YEARS"] / df["AGE_YEARS"]

# Vérification
print(df[["AGE_YEARS", "EMPLOYED_YEARS", "EMPLOYMENT_TO_AGE_RATIO"]].describe())

       AGE_YEARS  EMPLOYED_YEARS  EMPLOYMENT_TO_AGE_RATIO
count 356255.000      291607.000               291607.000
mean      43.448           6.566                    0.160
std       11.941           6.396                    0.135
min       20.000          -0.000                   -0.000
25%       34.000           2.137                    0.058
50%       43.000           4.556                    0.122
75%       53.000           8.767                    0.224
max       69.000          49.074                    0.732


Tout est cohérent :

- AGE_YEARS : 20 à 69 ans, moyenne 43 ans. Plage réaliste pour des demandeurs de crédit.
- EMPLOYED_YEARS : 0 à 49 ans, moyenne 6.5 ans. Le max de 49 ans est rare mais plausible.
- EMPLOYMENT_TO_AGE_RATIO : 0 à 0.73, moyenne 0.16. Un client qui a passé 73% de sa vie chez le même employeur est très stable.

In [8]:
# Traitement des scores externes
ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

# Agrégats des 3 scores externes
df["EXT_SOURCE_MEAN"] = df[ext_cols].mean(axis=1)
df["EXT_SOURCE_MIN"]  = df[ext_cols].min(axis=1)
df["EXT_SOURCE_STD"]  = df[ext_cols].std(axis=1)

# Flag : nombre de scores externes manquants par client
df["EXT_SOURCE_MISSING_COUNT"] = df[ext_cols].isna().sum(axis=1)

# Vérification
print(df[["EXT_SOURCE_MEAN", "EXT_SOURCE_MIN", 
          "EXT_SOURCE_STD", "EXT_SOURCE_MISSING_COUNT"]].describe())

       EXT_SOURCE_MEAN  EXT_SOURCE_MIN  EXT_SOURCE_STD  \
count       356076.000      356076.000      315305.000   
mean             0.509           0.398           0.151   
std              0.148           0.186           0.099   
min              0.000           0.000           0.000   
25%              0.414           0.254           0.073   
50%              0.524           0.401           0.136   
75%              0.621           0.550           0.214   
max              0.879           0.879           0.652   

       EXT_SOURCE_MISSING_COUNT  
count                356255.000  
mean                      0.742  
std                       0.651  
min                       0.000  
25%                       0.000  
50%                       1.000  
75%                       1.000  
max                       3.000  


Tout est cohérent :

- EXT_SOURCE_MEAN/MIN : plage 0 à 0.879, bien dans l'intervalle [0,1] attendu.
- EXT_SOURCE_STD : 315 305 valeurs seulement car on a besoin d'au moins 2 scores non-NaN pour calculer un écart-type. Normal.
- EXT_SOURCE_MISSING_COUNT : moyenne à 0.74, médiane à 1 — la moitié des clients manquent au moins un score externe. Max à 3 = quelques clients sans aucun score externe.

In [9]:
# Features binaires à partir de variables numériques
# HAS_CAR : OWN_CAR_AGE manquant = pas de voiture
df["HAS_CAR"] = df["OWN_CAR_AGE"].notna().astype(int)

# Flag manquants informatifs pour EXT_SOURCE_1 (56% manquants)
df["EXT_SOURCE_1_MISSING"] = df["EXT_SOURCE_1"].isna().astype(int)
df["EXT_SOURCE_3_MISSING"] = df["EXT_SOURCE_3"].isna().astype(int)

# Vérification
print(df[["HAS_CAR", "EXT_SOURCE_1_MISSING", "EXT_SOURCE_3_MISSING"]].mean().round(3))

HAS_CAR                0.340
EXT_SOURCE_1_MISSING   0.544
EXT_SOURCE_3_MISSING   0.195
dtype: float64


Cohérent avec l'EDA :

- HAS_CAR : 34% des clients ont une voiture
- EXT_SOURCE_1_MISSING : 54.4% — correspond aux ~56% observés dans le notebook 01
- EXT_SOURCE_3_MISSING : 19.5% — correspond aux ~19.8% observés

In [10]:
# Sélection des variables catégorielles
cat_cols = df.select_dtypes(include="str").columns.tolist()
print(f"Colonnes catégorielles : {len(cat_cols)}")
print(cat_cols)

# Identifier binary vs multiclass
binary_cols     = [c for c in cat_cols if df[c].nunique() == 2]
multiclass_cols = [c for c in cat_cols if df[c].nunique() > 2]

print(f"\nBinaires     : {len(binary_cols)}")
print(f"Multiclasses : {len(multiclass_cols)}")

Colonnes catégorielles : 16
['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']

Binaires     : 4
Multiclasses : 12


In [11]:
# Encodage des variables catégorielles
# Binaires : Label Encoding (0/1)
# Ces 4 colonnes ont exactement 2 modalités, l'ordre n'a pas d'importance
for col in binary_cols:
    df[col] = pd.factorize(df[col])[0]

# Multiclasses : One-Hot Encoding
# drop_first=True évite la multicolinéarité parfaite
df = pd.get_dummies(df, columns=multiclass_cols, drop_first=True)
df = df.copy()  # défragmente après ajout de colonnes

print(f"Shape après encodage : {df.shape}")
print(f"Nouvelles colonnes créées : {df.shape[1] - 123}")

Shape après encodage : (356255, 242)
Nouvelles colonnes créées : 119


119 nouvelles colonnes créées, principalement par ORGANIZATION_TYPE (58 modalités) et les autres variables multiclasses. C'est attendu.

In [12]:
# Imputation des valeurs manquantes
# Colonnes numériques — imputation par la médiane
# On calcule la médiane sur le train uniquement pour éviter le data leakage
train_mask = df["is_train"] == 1

num_cols = df.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c not in ["TARGET", "is_train", "SK_ID_CURR"]]

for col in num_cols:
    median_val = df.loc[train_mask, col].median()
    df[col] = df[col].fillna(median_val)

# Vérification
remaining_na = df.isnull().sum().sum()
print(f"Valeurs manquantes restantes : {remaining_na}")
print(f"Colonnes encore avec NaN : {df.isnull().sum()[df.isnull().sum() > 0]}")

Valeurs manquantes restantes : 48744
Colonnes encore avec NaN : TARGET    48744
dtype: int64


Parfait — les seuls NaN restants sont dans TARGET pour les 48 744 lignes du test, ce qui est exactement ce qu'on attend.

In [13]:
# Séparation finale des datasets train et test
train_final = df[df["is_train"] == 1].drop("is_train", axis=1)
test_final  = df[df["is_train"] == 0].drop(["is_train", "TARGET"], axis=1)

print(f"Train final : {train_final.shape}")
print(f"Test final  : {test_final.shape}")

# Sauvegarde
train_final.to_csv(DATA_PROCESSED / "train_application.csv", index=False)
test_final.to_csv(DATA_PROCESSED / "test_application.csv", index=False)

print("\nFichiers sauvegardés dans data/processed/")

Train final : (307511, 241)
Test final  : (48744, 240)

Fichiers sauvegardés dans data/processed/


On attaque maintenant les tables secondaires. L'ordre logique :

- bureau + bureau_balance — historique crédit externe
- previous_application — demandes passées
- POS_CASH_balance, installments_payments, credit_card_balance — historique interne

L'objectif pour chaque table : agréger au niveau SK_ID_CURR pour pouvoir fusionner avec notre dataset principal.

In [14]:
# Chargement des data du Credit Bureau
bureau          = pd.read_csv(DATA_RAW / "bureau.csv")
bureau_balance  = pd.read_csv(DATA_RAW / "bureau_balance.csv")

print(f"bureau          : {bureau.shape}")
print(f"bureau_balance  : {bureau_balance.shape}")

bureau          : (1716428, 17)
bureau_balance  : (27299925, 3)


In [15]:
# Exploration rapide du bureau
print("=== BUREAU ===")
print(bureau.dtypes)
print()
print(bureau.describe().T[["min", "max", "mean"]])
print()
print("=== BUREAU_BALANCE ===")
print(bureau_balance.dtypes)
print()
print(bureau_balance["STATUS"].value_counts())

=== BUREAU ===
SK_ID_CURR                  int64
SK_ID_BUREAU                int64
CREDIT_ACTIVE                 str
CREDIT_CURRENCY               str
DAYS_CREDIT                 int64
CREDIT_DAY_OVERDUE          int64
DAYS_CREDIT_ENDDATE       float64
DAYS_ENDDATE_FACT         float64
AMT_CREDIT_MAX_OVERDUE    float64
CNT_CREDIT_PROLONG          int64
AMT_CREDIT_SUM            float64
AMT_CREDIT_SUM_DEBT       float64
AMT_CREDIT_SUM_LIMIT      float64
AMT_CREDIT_SUM_OVERDUE    float64
CREDIT_TYPE                   str
DAYS_CREDIT_UPDATE          int64
AMT_ANNUITY               float64
dtype: object

                                min           max        mean
SK_ID_CURR               100001.000    456255.000  278214.934
SK_ID_BUREAU            5000000.000   6843457.000 5924434.489
DAYS_CREDIT               -2922.000         0.000   -1142.108
CREDIT_DAY_OVERDUE            0.000      2792.000       0.818
DAYS_CREDIT_ENDDATE      -42060.000     31199.000     510.517
DAYS_ENDDATE_FACT   

Plusieurs éléments importants à noter.
Dans bureau :

- DAYS_CREDIT_ENDDATE a un max à 31 199 — des crédits qui se terminent dans le futur, normal
- AMT_CREDIT_MAX_OVERDUE max à 115M et AMT_CREDIT_SUM max à 585M — valeurs extrêmes à surveiller
- DAYS_CREDIT_UPDATE a un max à 372 — valeur positive, à investiguer

Dans bureau_balance, la colonne STATUS :

- C = Closed (crédit clôturé)
- X = Unknown
- 0 = No DPD (no days past due — remboursement normal)
- 1 à 5 = nombre de mois de retard (1 = 1-30 jours, 5 = 5 mois+)

In [16]:
# Agrégation bureau_balance → bureau
# Créer des features à partir de STATUS
bureau_balance["STATUS_OVERDUE"] = bureau_balance["STATUS"].isin(["1","2","3","4","5"]).astype(int) # On crée une colonne binaire : 1 si le statut est un retard (1 à 5), 0 sinon (C, X, 0). C'est une transformation métier — on traduit les codes en signal de risque.

bb_agg = bureau_balance.groupby("SK_ID_BUREAU").agg(
    BB_COUNT            = ("MONTHS_BALANCE", "count"),
    BB_OVERDUE_COUNT    = ("STATUS_OVERDUE", "sum"),
    BB_OVERDUE_RATE     = ("STATUS_OVERDUE", "mean"),
).reset_index() 
                # On agrège au niveau SK_ID_BUREAU — chaque crédit bureau devient une seule ligne avec :
                    # BB_COUNT : combien de mois d'historique on a pour ce crédit
                    # BB_OVERDUE_COUNT : combien de mois en retard
                    # BB_OVERDUE_RATE : taux de mois en retard (BB_OVERDUE_COUNT / BB_COUNT)

print(f"bureau_balance agrégé : {bb_agg.shape}")
print(bb_agg.describe())

bureau_balance agrégé : (817395, 4)
       SK_ID_BUREAU   BB_COUNT  BB_OVERDUE_COUNT  BB_OVERDUE_RATE
count    817395.000 817395.000        817395.000       817395.000
mean    6022974.241     33.399             0.420            0.014
std      500713.769     25.795             2.275            0.059
min     5001709.000      1.000             0.000            0.000
25%     5700075.500     13.000             0.000            0.000
50%     6061126.000     26.000             0.000            0.000
75%     6430105.500     48.000             0.000            0.000
max     6842888.000     97.000            97.000            1.000


817 395 crédits uniques après agrégation. Le taux de retard moyen est de 1.4% — cohérent avec un portefeuille majoritairement sain

In [17]:
# Jointure sur SK_ID_BUREAU
bureau = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")

print(f"Bureau enrichi : {bureau.shape}")
print(f"NaN BB_COUNT : {bureau['BB_COUNT'].isna().sum()} "
      f"({bureau['BB_COUNT'].isna().mean()*100:.1f}%)")

Bureau enrichi : (1716428, 20)
NaN BB_COUNT : 942074 (54.9%)


54.9% des crédits bureau n'ont pas d'historique dans bureau_balance. C'est normal — bureau_balance ne couvre que les crédits actifs ou récemment clôturés, pas tous les crédits historiques.

In [18]:
# Agrégation bureau → SK_ID_CURR
bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    BUREAU_CREDIT_COUNT        = ("SK_ID_BUREAU", "count"),
    BUREAU_ACTIVE_COUNT        = ("CREDIT_ACTIVE", lambda x: (x == "Active").sum()),
    BUREAU_CLOSED_COUNT        = ("CREDIT_ACTIVE", lambda x: (x == "Closed").sum()),
    BUREAU_DAYS_CREDIT_MEAN    = ("DAYS_CREDIT", "mean"),
    BUREAU_DAYS_CREDIT_MIN     = ("DAYS_CREDIT", "min"),
    BUREAU_CREDIT_SUM          = ("AMT_CREDIT_SUM", "sum"),
    BUREAU_CREDIT_DEBT_SUM     = ("AMT_CREDIT_SUM_DEBT", "sum"),
    BUREAU_CREDIT_OVERDUE_MAX  = ("AMT_CREDIT_SUM_OVERDUE", "max"),
    BUREAU_OVERDUE_RATE_MEAN   = ("BB_OVERDUE_RATE", "mean"),
    BUREAU_OVERDUE_COUNT_SUM   = ("BB_OVERDUE_COUNT", "sum"),
).reset_index()

print(f"Bureau agrégé au niveau client : {bureau_agg.shape}")
print(bureau_agg.describe().T[["min", "max", "mean"]])

Bureau agrégé au niveau client : (305811, 11)
                                   min            max        mean
SK_ID_CURR                  100001.000     456255.000  278047.300
BUREAU_CREDIT_COUNT              1.000        116.000       5.613
BUREAU_ACTIVE_COUNT              0.000         32.000       2.062
BUREAU_CLOSED_COUNT              0.000        108.000       3.529
BUREAU_DAYS_CREDIT_MEAN      -2922.000          0.000   -1083.802
BUREAU_DAYS_CREDIT_MIN       -2922.000          0.000   -1764.363
BUREAU_CREDIT_SUM                0.000 1017957917.385 1992466.074
BUREAU_CREDIT_DEBT_SUM    -6981558.210  334498331.205  653914.190
BUREAU_CREDIT_OVERDUE_MAX        0.000    3756681.000     176.410
BUREAU_OVERDUE_RATE_MEAN         0.000          1.000       0.016
BUREAU_OVERDUE_COUNT_SUM         0.000        396.000       1.027


L'agrégation groupby("SK_ID_CURR").agg(...) regroupe toutes les lignes d'un même client en une seule ligne. Voici ce que chaque feature calcule :
- BUREAU_CREDIT_COUNT — count sur SK_ID_BUREAU
Nombre total de crédits externes que ce client a eu dans sa vie. Un client avec beaucoup de crédits passés est un profil expérimenté mais potentiellement surendetté.
- BUREAU_ACTIVE_COUNT et BUREAU_CLOSED_COUNT — lambda x: (x == "Active").sum()
On compte combien de crédits sont encore actifs vs clôturés. Un client avec beaucoup de crédits actifs simultanément est plus risqué.
- BUREAU_DAYS_CREDIT_MEAN et BUREAU_DAYS_CREDIT_MIN — mean et min sur DAYS_CREDIT
DAYS_CREDIT = date d'ouverture du crédit en jours depuis aujourd'hui (négatif). La moyenne donne l'ancienneté moyenne des crédits, le min donne le crédit le plus ancien.
- BUREAU_CREDIT_SUM — sum sur AMT_CREDIT_SUM
Encours total de crédit externe du client — somme de tous ses crédits bureau.
- BUREAU_CREDIT_DEBT_SUM — sum sur AMT_CREDIT_SUM_DEBT
Dette totale restante sur tous les crédits externes. C'est le signal de charge financière actuelle.
- BUREAU_CREDIT_OVERDUE_MAX — max sur AMT_CREDIT_SUM_OVERDUE
Montant maximum en retard sur un seul crédit. Un max élevé = incident significatif dans le passé.
- BUREAU_OVERDUE_RATE_MEAN — mean sur BB_OVERDUE_RATE
Taux moyen de mois en retard sur l'ensemble des crédits du client — vient de bureau_balance. C'est le signal comportemental le plus fort.
- BUREAU_OVERDUE_COUNT_SUM — sum sur BB_OVERDUE_COUNT
Nombre total de mois en retard cumulés sur tous les crédits. Un client avec 20 mois de retard cumulés est très différent d'un client avec 0.

305 811 clients sur 307 511 ont un historique bureau — 1 700 clients sans aucun crédit externe, ce qui est normal.
Quelques observations sur les valeurs :

- BUREAU_CREDIT_COUNT max à 116 — un client avec 116 crédits externes, cas extrême
- BUREAU_CREDIT_DEBT_SUM min négatif (-6.9M) — anomalie à surveiller, dette négative n'a pas de sens métier
- BUREAU_OVERDUE_RATE_MEAN moyenne à 1.6% — cohérent avec ce qu'on avait vu dans bureau_balance

In [19]:
# Jointure bureau_agg → df

df = df.merge(bureau_agg, on="SK_ID_CURR", how="left")

# Les clients sans historique bureau ont des NaN — imputer par 0
bureau_feature_cols = [c for c in bureau_agg.columns if c != "SK_ID_CURR"]
df[bureau_feature_cols] = df[bureau_feature_cols].fillna(0)

print(f"Dataset après ajout bureau : {df.shape}")
print(f"NaN restants : {df.isnull().sum().sum()}")

Dataset après ajout bureau : (356255, 252)
NaN restants : 48744


In [20]:
# Exploration rapide du dataset previous_application
prev = pd.read_csv(DATA_RAW / "previous_application.csv")

print(f"previous_application : {prev.shape}")
print()
print(prev.dtypes.value_counts())
print()
print(prev.describe().T[["min", "max", "mean"]])

previous_application : (1670214, 37)

str        16
float64    15
int64       6
Name: count, dtype: int64

                                  min         max        mean
SK_ID_PREV                1000001.000 2845382.000 1923089.135
SK_ID_CURR                 100001.000  456255.000  278357.174
AMT_ANNUITY                     0.000  418058.145   15955.121
AMT_APPLICATION                 0.000 6905160.000  175233.860
AMT_CREDIT                      0.000 6905160.000  196114.021
AMT_DOWN_PAYMENT               -0.900 3060045.000    6697.402
AMT_GOODS_PRICE                 0.000 6905160.000  227847.279
HOUR_APPR_PROCESS_START         0.000      23.000      12.484
NFLAG_LAST_APPL_IN_DAY          0.000       1.000       0.996
RATE_DOWN_PAYMENT              -0.000       1.000       0.080
RATE_INTEREST_PRIMARY           0.035       1.000       0.188
RATE_INTEREST_PRIVILEGED        0.373       1.000       0.774
DAYS_DECISION               -2922.000      -1.000    -880.680
SELLERPLACE_AREA         

On retrouve immédiatement la valeur sentinelle 365 243 dans plusieurs colonnes DAYS_ — DAYS_FIRST_DRAWING, DAYS_FIRST_DUE, DAYS_LAST_DUE_1ST_VERSION, DAYS_LAST_DUE, DAYS_TERMINATION. Cette fois on la reconnaît sans investigation — ce sont des demandes refusées ou annulées qui n'ont pas de dates réelles.
On note aussi AMT_DOWN_PAYMENT avec un min négatif (-0.9) — anomalie mineure à corriger.

In [21]:
# Remplacer 365243 par NaN dans toutes les colonnes DAYS_
days_cols_prev = ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", 
                  "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", 
                  "DAYS_TERMINATION"]

for col in days_cols_prev:
    prev[col] = prev[col].replace(365243, np.nan)

# Corriger AMT_DOWN_PAYMENT négatif
prev["AMT_DOWN_PAYMENT"] = prev["AMT_DOWN_PAYMENT"].clip(lower=0)

# Vérification
print("Après nettoyage :")
print(prev[days_cols_prev].describe().T[["min", "max"]])
print(f"\nAMT_DOWN_PAYMENT min : {prev['AMT_DOWN_PAYMENT'].min()}")

Après nettoyage :
                                min      max
DAYS_FIRST_DRAWING        -2922.000   -2.000
DAYS_FIRST_DUE            -2892.000   -2.000
DAYS_LAST_DUE_1ST_VERSION -2801.000 2389.000
DAYS_LAST_DUE             -2889.000   -2.000
DAYS_TERMINATION          -2874.000   -2.000

AMT_DOWN_PAYMENT min : 0.0


Les 365 243 sont remplacés, AMT_DOWN_PAYMENT est corrigé.
DAYS_LAST_DUE_1ST_VERSION a un max à 2389 — des échéances dans le futur, ce qui est normal pour des crédits encore actifs au moment de la collecte des données.

In [22]:
# Agrégation previous_application → SK_ID_CURR
prev_agg = prev.groupby("SK_ID_CURR").agg(
    PREV_COUNT                = ("SK_ID_PREV", "count"),
    PREV_APPROVED_COUNT       = ("NAME_CONTRACT_STATUS", lambda x: (x == "Approved").sum()),
    PREV_REFUSED_COUNT        = ("NAME_CONTRACT_STATUS", lambda x: (x == "Refused").sum()),
    PREV_AMT_CREDIT_MEAN      = ("AMT_CREDIT", "mean"),
    PREV_AMT_CREDIT_MAX       = ("AMT_CREDIT", "max"),
    PREV_AMT_ANNUITY_MEAN     = ("AMT_ANNUITY", "mean"),
    PREV_AMT_DOWN_PAYMENT_MAX = ("AMT_DOWN_PAYMENT", "max"),
    PREV_DAYS_DECISION_MEAN   = ("DAYS_DECISION", "mean"),
    PREV_DAYS_DECISION_MIN    = ("DAYS_DECISION", "min"),
    PREV_CNT_PAYMENT_MEAN     = ("CNT_PAYMENT", "mean"),
    PREV_RATE_DOWN_PAYMENT    = ("RATE_DOWN_PAYMENT", "mean"),
).reset_index()

# Ratio approbations
PREV_APPROVED_COUNT = prev_agg["PREV_APPROVED_COUNT"]
PREV_COUNT          = prev_agg["PREV_COUNT"]
prev_agg["PREV_APPROVAL_RATE"] = PREV_APPROVED_COUNT / PREV_COUNT

print(f"previous_application agrégé : {prev_agg.shape}")
print(prev_agg.describe().T[["min", "max", "mean"]])

previous_application agrégé : (338857, 13)
                                 min         max       mean
SK_ID_CURR                100001.000  456255.000 278149.910
PREV_COUNT                     1.000      77.000      4.929
PREV_APPROVED_COUNT            0.000      27.000      3.060
PREV_REFUSED_COUNT             0.000      68.000      0.858
PREV_AMT_CREDIT_MEAN           0.000 4050000.000 170331.849
PREV_AMT_CREDIT_MAX            0.000 6905160.000 416755.490
PREV_AMT_ANNUITY_MEAN          0.000  300425.445  14656.029
PREV_AMT_DOWN_PAYMENT_MAX      0.000 3060045.000  11767.142
PREV_DAYS_DECISION_MEAN    -2922.000      -2.000   -919.289
PREV_DAYS_DECISION_MIN     -2922.000      -2.000  -1542.694
PREV_CNT_PAYMENT_MEAN          0.000      72.000     14.533
PREV_RATE_DOWN_PAYMENT        -0.000       0.990      0.081
PREV_APPROVAL_RATE             0.000       1.000      0.744


338 857 clients ont un historique de demandes passées. PREV_APPROVAL_RATE moyenne à 74.4% — les trois quarts des demandes passées sont approuvées en moyenne.

In [23]:
# Jointure previous_application → df
df = df.merge(prev_agg, on="SK_ID_CURR", how="left")

prev_feature_cols = [c for c in prev_agg.columns if c != "SK_ID_CURR"]
df[prev_feature_cols] = df[prev_feature_cols].fillna(0)

print(f"Dataset après ajout previous_application : {df.shape}")
print(f"NaN restants : {df.isnull().sum().sum()}")

Dataset après ajout previous_application : (356255, 264)
NaN restants : 48744


264 colonnes, toujours 48 744 NaN uniquement dans TARGET.
On attaque les 3 dernières tables en suivant la même logique. Elles descendent toutes de previous_application via SK_ID_PREV, donc on agrège d'abord au niveau SK_ID_PREV, puis au niveau SK_ID_CURR

In [24]:
# Exploration rapide des données de crédit revolving
pos_cash     = pd.read_csv(DATA_RAW / "POS_CASH_balance.csv")
installments = pd.read_csv(DATA_RAW / "installments_payments.csv")
credit_card  = pd.read_csv(DATA_RAW / "credit_card_balance.csv")

print(f"POS_CASH_balance       : {pos_cash.shape}")
print(f"installments_payments  : {installments.shape}")
print(f"credit_card_balance    : {credit_card.shape}")

POS_CASH_balance       : (10001358, 8)
installments_payments  : (13605401, 8)
credit_card_balance    : (3840312, 23)


In [25]:
print("=== POS_CASH_balance ===")
print(pos_cash.dtypes)
print(pos_cash.describe().T[["min", "max", "mean"]])

print("\n=== installments_payments ===")
print(installments.dtypes)
print(installments.describe().T[["min", "max", "mean"]])

print("\n=== credit_card_balance ===")
print(credit_card.dtypes)
print(credit_card.describe().T[["min", "max", "mean"]])

=== POS_CASH_balance ===
SK_ID_PREV                 int64
SK_ID_CURR                 int64
MONTHS_BALANCE             int64
CNT_INSTALMENT           float64
CNT_INSTALMENT_FUTURE    float64
NAME_CONTRACT_STATUS         str
SK_DPD                     int64
SK_DPD_DEF                 int64
dtype: object
                              min         max        mean
SK_ID_PREV            1000001.000 2843499.000 1903216.599
SK_ID_CURR             100001.000  456255.000  278403.863
MONTHS_BALANCE            -96.000      -1.000     -35.013
CNT_INSTALMENT              1.000      92.000      17.090
CNT_INSTALMENT_FUTURE       0.000      85.000      10.484
SK_DPD                      0.000    4231.000      11.607
SK_DPD_DEF                  0.000    3595.000       0.654

=== installments_payments ===
SK_ID_PREV                  int64
SK_ID_CURR                  int64
NUM_INSTALMENT_VERSION    float64
NUM_INSTALMENT_NUMBER       int64
DAYS_INSTALMENT           float64
DAYS_ENTRY_PAYMENT        float6

POS_CASH — SK_DPD max à 4231 jours de retard, signal de risque fort. SK_DPD_DEF est la version déflaté (ajustée).

installments — DAYS_ENTRY_PAYMENT min à -4921 vs DAYS_INSTALMENT min à -2922 — des paiements enregistrés avant la date d'échéance prévue, possible. On peut créer un feature PAYMENT_DELAY = DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT — positif = en retard, négatif = en avance.

credit_card — plusieurs colonnes avec valeurs négatives (AMT_BALANCE, AMT_DRAWINGS_ATM_CURRENT) — des remboursements ou corrections comptables, pas des anomalies à corriger.

In [26]:
# Agrégation POS_CASH
pos_agg = pos_cash.groupby("SK_ID_CURR").agg(
    POS_COUNT             = ("SK_ID_PREV", "count"),
    POS_MONTHS_BALANCE    = ("MONTHS_BALANCE", "mean"),
    POS_CNT_INSTALMENT    = ("CNT_INSTALMENT", "mean"),
    POS_DPD_MEAN          = ("SK_DPD", "mean"),
    POS_DPD_MAX           = ("SK_DPD", "max"),
    POS_DPD_DEF_MEAN      = ("SK_DPD_DEF", "mean"),
    POS_COMPLETED_COUNT   = ("NAME_CONTRACT_STATUS", 
                             lambda x: (x == "Completed").sum()),
).reset_index()

print(f"POS_CASH agrégé : {pos_agg.shape}")
print(pos_agg.describe().T[["min", "max", "mean"]])

POS_CASH agrégé : (337252, 8)
                           min        max       mean
SK_ID_CURR          100001.000 456255.000 278163.133
POS_COUNT                1.000    295.000     29.655
POS_MONTHS_BALANCE     -96.000     -1.000    -31.873
POS_CNT_INSTALMENT       1.000     72.000     14.655
POS_DPD_MEAN             0.000   2622.078      4.296
POS_DPD_MAX              0.000   4231.000     15.294
POS_DPD_DEF_MEAN         0.000   1740.554      0.225
POS_COMPLETED_COUNT      0.000     86.000      2.209


In [27]:
# Agrégation installments
# Feature clé : délai de paiement (positif = retard, négatif = en avance)
installments["PAYMENT_DELAY"] = (installments["DAYS_ENTRY_PAYMENT"] 
                                 - installments["DAYS_INSTALMENT"])

# Ratio paiement effectué vs prévu
installments["PAYMENT_RATIO"] = (installments["AMT_PAYMENT"] 
                                 / installments["AMT_INSTALMENT"].replace(0, np.nan))

inst_agg = installments.groupby("SK_ID_CURR").agg(
    INST_COUNT              = ("SK_ID_PREV", "count"),
    INST_PAYMENT_DELAY_MEAN = ("PAYMENT_DELAY", "mean"),
    INST_PAYMENT_DELAY_MAX  = ("PAYMENT_DELAY", "max"),
    INST_PAYMENT_RATIO_MEAN = ("PAYMENT_RATIO", "mean"),
    INST_PAYMENT_RATIO_MIN  = ("PAYMENT_RATIO", "min"),
    INST_AMT_PAYMENT_SUM    = ("AMT_PAYMENT", "sum"),
    INST_AMT_INSTALMENT_SUM = ("AMT_INSTALMENT", "sum"),
).reset_index()

print(f"installments agrégé : {inst_agg.shape}")
print(inst_agg.describe().T[["min", "max", "mean"]])

installments agrégé : (339587, 8)
                               min          max       mean
SK_ID_CURR              100001.000   456255.000 278154.892
INST_COUNT                   1.000      372.000     40.065
INST_PAYMENT_DELAY_MEAN   -295.000     1884.205    -11.258
INST_PAYMENT_DELAY_MAX    -156.000     2884.000     15.695
INST_PAYMENT_RATIO_MEAN      0.333     8482.446      1.360
INST_PAYMENT_RATIO_MIN       0.000        1.970      0.597
INST_AMT_PAYMENT_SUM         0.000 32689281.510 690494.226
INST_AMT_INSTALMENT_SUM      0.000 32479781.265 683136.949


Observations intéressantes :

- INST_PAYMENT_DELAY_MEAN moyenne à -11.3 jours — les clients paient en avance en moyenne, bon signe
- INST_PAYMENT_DELAY_MAX moyenne à 15.7 jours — mais certains ont des retards ponctuels
- INST_PAYMENT_RATIO_MEAN max à 8482 — des surpaiements massifs, probablement des remboursements anticipés

In [28]:
# Agrégation credit_card
cc_agg = credit_card.groupby("SK_ID_CURR").agg(
    CC_COUNT                = ("SK_ID_PREV", "count"),
    CC_AMT_BALANCE_MEAN     = ("AMT_BALANCE", "mean"),
    CC_AMT_BALANCE_MAX      = ("AMT_BALANCE", "max"),
    CC_AMT_CREDIT_LIMIT     = ("AMT_CREDIT_LIMIT_ACTUAL", "mean"),
    CC_AMT_DRAWINGS_MEAN    = ("AMT_DRAWINGS_CURRENT", "mean"),
    CC_AMT_PAYMENT_MEAN     = ("AMT_PAYMENT_CURRENT", "mean"),
    CC_DPD_MEAN             = ("SK_DPD", "mean"),
    CC_DPD_MAX              = ("SK_DPD", "max"),
    CC_DPD_DEF_MEAN         = ("SK_DPD_DEF", "mean"),
    CC_UTILIZATION_RATE     = ("AMT_BALANCE", "mean"),
).reset_index()

# Taux d'utilisation = solde moyen / limite de crédit
cc_agg["CC_UTILIZATION_RATE"] = (cc_agg["CC_AMT_BALANCE_MEAN"] / 
                                  cc_agg["CC_AMT_CREDIT_LIMIT"].replace(0, np.nan))

print(f"credit_card agrégé : {cc_agg.shape}")
print(cc_agg.describe().T[["min", "max", "mean"]])

credit_card agrégé : (103558, 11)
                            min         max       mean
SK_ID_CURR           100006.000  456250.000 278381.458
CC_COUNT                  1.000     192.000     37.084
CC_AMT_BALANCE_MEAN   -2930.233  928686.324  69973.192
CC_AMT_BALANCE_MAX        0.000 1505902.185 142297.932
CC_AMT_CREDIT_LIMIT       0.000 1350000.000 207320.670
CC_AMT_DRAWINGS_MEAN    -17.578 1616206.320  13566.784
CC_AMT_PAYMENT_MEAN       0.000 1593111.184  17934.679
CC_DPD_MEAN               0.000    1635.685      4.107
CC_DPD_MAX                0.000    3260.000     16.402
CC_DPD_DEF_MEAN           0.000    1635.685      0.152
CC_UTILIZATION_RATE      -0.085       2.139      0.324


CC_UTILIZATION_RATE moyenne à 32.4% — cohérent, un taux d'utilisation sain est généralement sous 30-35%. Le max à 2.139 signifie que certains clients dépassent leur limite de crédit.

In [29]:
# Jointure des agrégats POS_CASH, installments, credit_card → df
for agg, name in [(pos_agg, "POS_CASH"), 
                  (inst_agg, "installments"), 
                  (cc_agg, "credit_card")]:
    df = df.merge(agg, on="SK_ID_CURR", how="left")
    feature_cols = [c for c in agg.columns if c != "SK_ID_CURR"]
    df[feature_cols] = df[feature_cols].fillna(0)
    print(f"Après ajout {name} : {df.shape}")

print(f"\nNaN restants : {df.isnull().sum().sum()}")

Après ajout POS_CASH : (356255, 271)
Après ajout installments : (356255, 278)
Après ajout credit_card : (356255, 288)

NaN restants : 48744


288 colonnes au total, toujours 48 744 NaN uniquement dans TARGET. Toutes les jointures sont propres.

In [30]:
# Séparation finale des datasets train et test
train_final = df[df["is_train"] == 1].drop("is_train", axis=1)
test_final  = df[df["is_train"] == 0].drop(["is_train", "TARGET"], axis=1)

print(f"Train final : {train_final.shape}")
print(f"Test final  : {test_final.shape}")

# Sauvegarde
train_final.to_csv(DATA_PROCESSED / "train_final.csv", index=False)
test_final.to_csv(DATA_PROCESSED / "test_final.csv", index=False)

print("\nFichiers sauvegardés :")
print(f"  data/processed/train_final.csv")
print(f"  data/processed/test_final.csv")

Train final : (307511, 287)
Test final  : (48744, 286)

Fichiers sauvegardés :
  data/processed/train_final.csv
  data/processed/test_final.csv


287 features pour le train, 286 pour le test (sans TARGET). On est passé de 122 colonnes brutes à 287 features enrichies.
Récapitulatif de qui a été construit :

- Application : 122 → 241 colonnes (features temporelles, EXT_SOURCE, binaires, encodage)
- Bureau + bureau_balance : +11 features (historique crédit externe)
- Previous application : +13 features (demandes passées, taux d'approbation)
- POS_CASH : +7 features (retards POS)
- Installments : +7 features (délais et ratios de paiement)
- Credit card : +10 features (utilisation carte, retards)

## Conclusion

Ce notebook a permis de transformer les données brutes Home Credit en un jeu de données exploitable pour la modélisation.

Le travail a d’abord porté sur la table principale `application_train` / `application_test`. Les deux jeux ont été concaténés afin d’appliquer exactement les mêmes traitements au train et au test, puis plusieurs transformations ont été réalisées :
- correction de l’anomalie `DAYS_EMPLOYED = 365243` avec création d’un flag dédié ;
- création de variables temporelles plus lisibles comme `AGE_YEARS`, `EMPLOYED_YEARS` et `EMPLOYMENT_TO_AGE_RATIO` ;
- enrichissement autour des scores externes avec des agrégats (`EXT_SOURCE_MEAN`, `EXT_SOURCE_MIN`, `EXT_SOURCE_STD`) et des indicateurs de valeurs manquantes ;
- ajout de variables binaires informatives comme `HAS_CAR`, `EXT_SOURCE_1_MISSING` et `EXT_SOURCE_3_MISSING`.

Les variables catégorielles ont ensuite été encodées selon leur nature :
- les variables binaires ont été converties en `0/1` par factorisation ;
- les variables à plusieurs modalités ont été transformées par one-hot encoding avec `drop_first=True`.

Les valeurs manquantes numériques ont été imputées à l’aide de la médiane calculée sur le train uniquement, afin d’éviter toute fuite de données.

Le notebook a ensuite enrichi la table principale grâce aux tables secondaires :
- `bureau` et `bureau_balance` pour l’historique de crédit externe ;
- `previous_application` pour les demandes passées ;
- `POS_CASH_balance`, `installments_payments` et `credit_card_balance` pour l’historique interne de paiement et d’utilisation du crédit.

Pour chacune de ces tables, des agrégats ont été calculés au niveau `SK_ID_CURR` afin de résumer le comportement du client sous forme de nouvelles features métier : nombre de crédits, retards, montants moyens, taux d’approbation, ratios de paiement, utilisation de la carte, etc.

Au final, le jeu de données est passé de **122 colonnes brutes** à **287 features pour le train** et **286 pour le test**. Les fichiers finaux sauvegardés dans `data/processed/` sont :
- `train_final.csv`
- `test_final.csv`

Ces deux jeux constituent la base finale utilisée dans le notebook de modélisation.
